# <center> <img src="../img/ITESOLogo.png" alt="ITESO" width="480" height="130"> </center>
# <center> **Departamento de Electrónica, Sistemas e Informática** </center>
---
## <center> **Big Data** </center>
---
### <center> **Spring 2026** </center>
---
### <center> **Examples on Structured Streaming (sockets)** </center>
---
**Profesor**: Pablo Camarillo Ramirez

# Create SparkSession

In [ ]:
import findspark
findspark.init()

from pcamarillor.spark_utils import SparkUtils

su = SparkUtils("Examples on Structured Streaming",
                   master_url="spark://spark-master:7077")

su.spark

# Create a data stream from a local socket

### Install netcat utility

In [ ]:
!apt-get update
!apt-get install -y netcat

### Connect Spark to the socket

In [ ]:
import pyspark.sql.functions as F

# Create the remote connection
lines = su.spark.readStream \
            .format("socket") \
            .option("host", "localhost") \
            .option("port", 9999) \
            .load()

# Perform some transformations to the input data (word counter)
words = lines.select(F.explode(F.split(lines.value, " ")).alias("word"))
word_count = words.groupBy("word").count()

# Send transformed data to the Sink
query = word_count.writeStream \
            .outputMode("complete") \
            .format("console") \
            .start()
query.awaitTermination(300)


In [ ]:
su.spark.stop()